In [2]:
import pandas as pd
import glob
import os

# 1. Detección automática de la ruta de datos
posibles_rutas = [
    "../datasets/final/",
    "datasets/final/",
    "./datasets/final/"
]

ruta_carpeta = None
archivos = []

for ruta in posibles_rutas:
    archivos_encontrados = glob.glob(os.path.join(ruta, "*_final.csv"))
    if len(archivos_encontrados) > 0:
        ruta_carpeta = ruta
        archivos = archivos_encontrados
        break

if not archivos:
    print("ERROR: No se encontraron archivos *_final.csv.")
    print("Por favor, verifica en qué carpeta están guardados tus CSV procesados.")
else:
    print(f"¡Carpeta encontrada! ({ruta_carpeta})")
    print(f"Se procesarán {len(archivos)} archivos CSV:")
    for a in archivos:
        print(f"   - {os.path.basename(a)}")

    # 2. Consolidación de archivos
    lista_dfs = []
    for archivo in archivos:
        df_temp = pd.read_csv(archivo)
        
        # Extraer nombre del producto desde el nombre del archivo
        nombre_base = os.path.basename(archivo)
        nombre_producto = nombre_base.split('(')[0].replace('_final.csv', '').strip()
        
        df_temp['producto'] = nombre_producto
        lista_dfs.append(df_temp)

    df_maestro = pd.concat(lista_dfs, ignore_index=True)
    df_maestro['fecdoc'] = pd.to_datetime(df_maestro['fecdoc'])

    # 3. Clasificación ABC (Pareto)
    df_ventas = df_maestro.groupby('producto')['cantidad'].sum().reset_index().sort_values(by='cantidad', ascending=False)
    df_ventas['porcentaje_acumulado'] = (df_ventas['cantidad'] / df_ventas['cantidad'].sum()).cumsum() * 100

    def clasificar_abc(pct):
        if pct <= 80:
            return 'Clase A (Alta Rotación)'
        elif pct <= 95:
            return 'Clase B (Rotación Media)'
        else:
            return 'Clase C (Baja Rotación)'

    df_ventas['categoria_abc'] = df_ventas['porcentaje_acumulado'].apply(clasificar_abc)

    # Inyectar clasificación ABC al dataset maestro
    df_maestro = df_maestro.merge(df_ventas[['producto', 'categoria_abc']], on='producto', how='left')

    # 4. Guardar dataset final para el Dashboard
    ruta_salida = os.path.join(os.path.dirname(os.path.normpath(ruta_carpeta)), "dataset_maestro_dashboard.csv")
    df_maestro.to_csv(ruta_salida, index=False)

    print("\n" + "="*50)
    print("¡DATASET MAESTRO GENERADO CON ÉXITO!")
    print("="*50)
    print(f"Guardado en: {os.path.abspath(ruta_salida)}")
    print(f"Total de registros: {len(df_maestro):,}")
    print(f"Productos procesados: {df_maestro['producto'].nunique()}")
    print("\nVistas primeras 5 filas:")
    display(df_maestro.head())

¡Carpeta encontrada! (../datasets/final/)
Se procesarán 14 archivos CSV:
   - AMOXICILINA 500MG(2024-2026)_final.csv
   - AZITROMICINA 500MG(2024-2026)_final.csv
   - CETIRIZINA 10MG(2024-2026)_final.csv
   - DIMENHIDRINATO 50MG(2024-2026)_final.csv
   - ENALAPRIL 10MG(2024-2026)_final.csv
   - IBUPROFENO TABX800(2024-2026)_final.csv
   - LOPERAMIDA 2MG(2024-2026)_final.csv
   - LORATADINA 10MG(2024-2026)_final.csv
   - LOSARTAN 50MG(2024-2026)I_final.csv
   - METFORMINA 850MG(2024-2026)_final.csv
   - NAPROXENO TABX550MG(2024-2026)_final.csv
   - OMEPRAZOL 20MG(2024-2026) I_final.csv
   - PARACETAMOL 500MG(2024-2026)_final.csv
   - VITAMINA C+ZINC+D3(2024-2026)_final.csv

¡DATASET MAESTRO GENERADO CON ÉXITO!
Guardado en: c:\Users\vicas\OneDrive\Escritorio\LF\Modelos_ML\datasets\dataset_maestro_dashboard.csv
Total de registros: 44,592
Productos procesados: 14

Vistas primeras 5 filas:


,codpro,prod,tipo,cantidad,unidades,total,fecdoc,sucursal,mes,estacion,producto,categoria_abc
0,108176,AMOXICILINA 500MG CJAX100CAP,PHARMA GENERICOS,0.0,10.0,3.0,2024-05-09 11:16:28,5,5,Otoño,AMOXICILINA 500MG,Clase C (Baja Rotación)
1,108176,AMOXICILINA 500MG CJAX100CAP,PHARMA GENERICOS,1.0,0.0,24.0,2024-05-22 11:44:45,5,5,Otoño,AMOXICILINA 500MG,Clase C (Baja Rotación)
2,108176,AMOXICILINA 500MG CJAX100CAP,PHARMA GENERICOS,0.0,2.0,0.6,2024-06-02 10:47:30,9,6,Invierno,AMOXICILINA 500MG,Clase C (Baja Rotación)
3,108176,AMOXICILINA 500MG CJAX100CAP,PHARMA GENERICOS,0.0,10.0,3.0,2024-06-02 11:43:30,9,6,Invierno,AMOXICILINA 500MG,Clase C (Baja Rotación)
4,108176,AMOXICILINA 500MG CJAX100CAP,PHARMA GENERICOS,0.0,10.0,3.0,2024-06-03 14:15:40,1,6,Invierno,AMOXICILINA 500MG,Clase C (Baja Rotación)
